# BUSI Notebook 3/3: Ablation Study + Statistical Significance

**Standalone** — only needs the BUSI dataset zip (same one from Notebook
1). It does **not** need Notebook 1's checkpoints; it trains everything
fresh here, including the full model, so results are self-consistent.

**Warning: this is the slowest notebook.** It trains the full model
plus 2 ablation variants plus 2 extra random-seed runs of the full model
(5 full training runs total). On a Colab GPU this can take **1.5-3
hours** depending on `EPOCHS`. Lower `EPOCHS` or `N_SEEDS` below if
you're short on time.

**Recommended:** Runtime -> Change runtime type -> GPU.


## 1. Setup

In [ ]:
!pip install -q torch torchvision pillow numpy scikit-learn

In [ ]:
import os, re, json, zipfile, random
from glob import glob
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
os.makedirs("outputs", exist_ok=True)


## 2. Load the dataset (same zip as Notebook 1)

In [ ]:
DATA_ZIP_PATH = "archive.zip"
EXTRACT_DIR = "busi_data"

if not os.path.exists(DATA_ZIP_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_ZIP_PATH}' not found. Please upload the BUSI dataset zip file:")
        uploaded = files.upload()
        DATA_ZIP_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"'{DATA_ZIP_PATH}' not found and not running in Colab.")

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}/")

candidates = glob(os.path.join(EXTRACT_DIR, "**", "Dataset_BUSI_with_GT"), recursive=True)
if not candidates:
    if all(os.path.isdir(os.path.join(EXTRACT_DIR, c)) for c in ["benign", "malignant", "normal"]):
        DATA_ROOT = EXTRACT_DIR
    else:
        raise FileNotFoundError("Could not locate Dataset_BUSI_with_GT folder inside the extracted zip.")
else:
    DATA_ROOT = candidates[0]

print("Dataset root:", DATA_ROOT)


## 3. Index + stratified split (identical seed/logic to Notebook 1)

In [ ]:
CLASS_NAMES = ["benign", "malignant", "normal"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def index_dataset(root_dir):
    records = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root_dir, cls)
        all_files = os.listdir(cls_dir)
        image_files = sorted(f for f in all_files if "_mask" not in f)
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_pattern = re.compile(rf"^{re.escape(stem)}_mask(_\d+)?\.png$")
            masks = sorted(os.path.join(cls_dir, f) for f in all_files if mask_pattern.match(f))
            records.append({"image": os.path.join(cls_dir, img_file), "masks": masks,
                             "label": CLASS_TO_IDX[cls], "class": cls})
    return records

def stratified_split(records, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class"]].append(r)
    train, val, test = [], [], []
    for cls, items in by_class.items():
        items = items.copy()
        rng.shuffle(items)
        n = len(items)
        n_val = int(round(n * val_frac))
        n_test = int(round(n * test_frac))
        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

records = index_dataset(DATA_ROOT)
train_recs, val_recs, test_recs = stratified_split(records)
print(f"Train: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}")


## 4. PyTorch Dataset + DataLoaders

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 16

def load_merged_mask(mask_paths, size=None):
    if len(mask_paths) == 0:
        return None
    from PIL import Image
    merged = None
    for p in mask_paths:
        m = np.array(Image.open(p).convert("L"))
        m = (m > 127).astype(np.uint8)
        merged = m if merged is None else np.maximum(merged, m)
    mask_img = Image.fromarray(merged * 255)
    if size is not None:
        mask_img = mask_img.resize(size, resample=Image.NEAREST)
    return mask_img

class BUSIDataset(Dataset):
    def __init__(self, records, img_size=256, augment=False):
        self.records = records
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        from PIL import Image
        rec = self.records[idx]
        size = (self.img_size, self.img_size)
        image = Image.open(rec["image"]).convert("RGB").resize(size, resample=Image.BILINEAR)
        mask_img = load_merged_mask(rec["masks"], size=size)
        if mask_img is None:
            mask_img = Image.new("L", size, 0)
        if self.augment:
            image, mask_img = self._augment(image, mask_img)
        image_t = TF.to_tensor(image)
        mask_t = (TF.to_tensor(mask_img) > 0.5).float()
        label_t = torch.tensor(rec["label"], dtype=torch.long)
        return image_t, mask_t, label_t

    def _augment(self, image, mask):
        if np.random.rand() < 0.5:
            image = TF.hflip(image); mask = TF.hflip(mask)
        if np.random.rand() < 0.5:
            image = TF.vflip(image); mask = TF.vflip(mask)
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-15, 15)
            image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)
        return image, mask

train_ds = BUSIDataset(train_recs, img_size=IMG_SIZE, augment=True)
val_ds = BUSIDataset(val_recs, img_size=IMG_SIZE, augment=False)
test_ds = BUSIDataset(test_recs, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 5. Model building blocks

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, channels))

    def forward(self, x):
        avg = F.adaptive_avg_pool2d(x, 1).flatten(1)
        mx = F.adaptive_max_pool2d(x, 1).flatten(1)
        attn = torch.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * attn.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class IdentityAttn(nn.Module):
    """No-op, used for the No-CBAM ablation."""
    def forward(self, x):
        return x

class DecoderBlock(nn.Module):
    """Full decoder block, with CBAM."""
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.attn = CBAM(out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.attn(x)

class DecoderBlockNoAttn(nn.Module):
    """Decoder block with attention replaced by identity (No-CBAM ablation)."""
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.attn = IdentityAttn()

    def forward(self, x, skip):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.attn(x)


## 6. Full model + ablation variants

In [ ]:
class BUSIMultiTaskNet(nn.Module):
    """Full model: CBAM decoder + mask-guided classification."""
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)
        self.cls_dropout = nn.Dropout(0.3)
        self.cls_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self.dec4(s4, s3); d3 = self.dec3(d4, s2); d2 = self.dec2(d3, s1); d1 = self.dec1(d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        seg_logits = self.seg_head(d0)
        mask_prob = torch.sigmoid(seg_logits.detach())
        mask_small = F.adaptive_avg_pool2d(mask_prob, s4.shape[-2:])
        weighted_feats = s4 * (1.0 + mask_small)
        pooled = F.adaptive_avg_pool2d(weighted_feats, 1).flatten(1)
        cls_logits = self.cls_fc(self.cls_dropout(pooled))
        return seg_logits, cls_logits


class AblationNoCBAM(nn.Module):
    """Ablation: decoder attention removed, mask-guided classification kept."""
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlockNoAttn(512, 256, 256)
        self.dec3 = DecoderBlockNoAttn(256, 128, 128)
        self.dec2 = DecoderBlockNoAttn(128, 64, 64)
        self.dec1 = DecoderBlockNoAttn(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)
        self.cls_dropout = nn.Dropout(0.3)
        self.cls_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self.dec4(s4, s3); d3 = self.dec3(d4, s2); d2 = self.dec2(d3, s1); d1 = self.dec1(d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        seg_logits = self.seg_head(d0)
        mask_prob = torch.sigmoid(seg_logits.detach())
        mask_small = F.adaptive_avg_pool2d(mask_prob, s4.shape[-2:])
        weighted_feats = s4 * (1.0 + mask_small)
        pooled = F.adaptive_avg_pool2d(weighted_feats, 1).flatten(1)
        cls_logits = self.cls_fc(self.cls_dropout(pooled))
        return seg_logits, cls_logits


class AblationNoMaskGuide(nn.Module):
    """Ablation: CBAM decoder kept, classification uses raw pooled features (no mask guidance)."""
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)
        self.cls_dropout = nn.Dropout(0.3)
        self.cls_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self.dec4(s4, s3); d3 = self.dec3(d4, s2); d2 = self.dec2(d3, s1); d1 = self.dec1(d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        seg_logits = self.seg_head(d0)
        # No mask guidance: plain global average pooling on raw encoder features
        pooled = F.adaptive_avg_pool2d(s4, 1).flatten(1)
        cls_logits = self.cls_fc(self.cls_dropout(pooled))
        return seg_logits, cls_logits


## 7. Losses, metrics, and the training-epoch function

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits).flatten(1)
    targets = targets.flatten(1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()

def dice_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1)
        return ((2 * intersection + eps) / (union + eps)).mean().item()

def iou_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
        return ((intersection + eps) / (union + eps)).mean().item()

def compute_class_weights(records, num_classes):
    counts = np.zeros(num_classes)
    for r in records:
        counts[r["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

class CombinedLoss(nn.Module):
    def __init__(self, seg_weight=1.0, cls_weight=1.0, class_weights=None):
        super().__init__()
        self.seg_weight = seg_weight
        self.cls_weight = cls_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, seg_logits, seg_target, cls_logits, cls_target):
        seg_loss = self.bce(seg_logits, seg_target) + dice_loss(seg_logits, seg_target)
        cls_loss = self.ce(cls_logits, cls_target)
        total = self.seg_weight * seg_loss + self.cls_weight * cls_loss
        return total, seg_loss.item(), cls_loss.item()

def run_multitask_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss = total_dice = total_iou = 0.0
    correct = n_samples = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks, labels in loader:
            images, masks, labels = images.to(device), masks.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            seg_logits, cls_logits = model(images)
            loss, seg_l, cls_l = criterion(seg_logits, masks, cls_logits, labels)
            if train:
                loss.backward(); optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            total_dice += dice_score(seg_logits, masks) * bs
            total_iou += iou_score(seg_logits, masks) * bs
            correct += (cls_logits.argmax(dim=1) == labels).sum().item()
            n_samples += bs
    return {"loss": total_loss / n_samples, "dice": total_dice / n_samples,
            "iou": total_iou / n_samples, "accuracy": correct / n_samples}

def evaluate_multitask_model_on_test(m, loader, device=DEVICE):
    m.eval()
    dice_list, iou_list, correct, n = [], [], 0, 0
    with torch.no_grad():
        for images, masks, labels in loader:
            images, masks, labels = images.to(device), masks.to(device), labels.to(device)
            seg_logits, cls_logits = m(images)
            dice_list.append(dice_score(seg_logits, masks))
            iou_list.append(iou_score(seg_logits, masks))
            correct += (cls_logits.argmax(dim=1) == labels).sum().item()
            n += images.size(0)
    return {"dice": float(np.mean(dice_list)), "iou": float(np.mean(iou_list)), "accuracy": correct / n}

EPOCHS = 40
LR = 1e-4
PRETRAINED = True
class_weights = compute_class_weights(train_recs, len(CLASS_NAMES)).to(DEVICE)
print("Class weights (benign, malignant, normal):", class_weights.tolist())


## 8. Train the full model (seed 42) — used as the reference point for both
the ablation table and the statistical significance summary


In [ ]:
def train_full_run(model_cls, name, seed, epochs=EPOCHS):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = model_cls(num_classes=len(CLASS_NAMES), pretrained=PRETRAINED).to(DEVICE)
    crit = CombinedLoss(class_weights=class_weights)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    best, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        run_multitask_epoch(m, train_loader, crit, opt, DEVICE, train=True)
        va = run_multitask_epoch(m, val_loader, crit, opt, DEVICE, train=False)
        sched.step(va["loss"])
        combined = va["dice"] + va["accuracy"]
        if combined > best:
            best = combined
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
            print(f"[{name}] Epoch {epoch}/{epochs} | val_dice {va['dice']:.4f} | val_acc {va['accuracy']:.4f}")
    m.load_state_dict(best_state)
    torch.save(m.state_dict(), f"outputs/best_{name}.pt")
    return m

print("Training full model (seed 42) ...")
full_model_seed42 = train_full_run(BUSIMultiTaskNet, "fullmodel_seed42", seed=42)
full_model_seed42_metrics = evaluate_multitask_model_on_test(full_model_seed42, test_loader)
print("Full model (seed 42) test metrics:", full_model_seed42_metrics)


## 9. Ablation study: train No-CBAM and No-Mask-Guidance variants

In [ ]:
print("Training ablation: No-CBAM ...")
ablation_no_cbam = train_full_run(AblationNoCBAM, "ablation_no_cbam", seed=42)
ablation_no_cbam_metrics = evaluate_multitask_model_on_test(ablation_no_cbam, test_loader)

print("\nTraining ablation: No-Mask-Guidance ...")
ablation_no_maskguide = train_full_run(AblationNoMaskGuide, "ablation_no_maskguide", seed=42)
ablation_no_maskguide_metrics = evaluate_multitask_model_on_test(ablation_no_maskguide, test_loader)

ablation_results = {
    "Full Model (CBAM + Mask-Guided)": full_model_seed42_metrics,
    "No-CBAM": ablation_no_cbam_metrics,
    "No-Mask-Guidance": ablation_no_maskguide_metrics,
}

print(f"\n{'Variant':<35}{'Dice':<10}{'IoU':<10}{'Accuracy':<10}")
print("-" * 65)
for name, m in ablation_results.items():
    print(f"{name:<35}{m['dice']:<10.4f}{m['iou']:<10.4f}{m['accuracy']:<10.4f}")

with open("outputs/ablation_results.json", "w") as f:
    json.dump(ablation_results, f, indent=2)


## 10. Statistical significance (multiple random seeds)

Reuses the seed-42 full-model result from Section 8, then trains 2 more
seeds of the full model, and reports **mean ± std** across all 3 seeds.

Lower `N_SEEDS` to 1 (i.e. just reuse Section 8's result, skip this
whole section) if you're very short on time — but 3 seeds is the
minimum most reviewers expect for a mean ± std claim.


In [ ]:
N_SEEDS = 3
EXTRA_SEEDS = [123, 2024][:max(0, N_SEEDS - 1)]

seed_results = {"dice": [full_model_seed42_metrics["dice"]],
                "iou": [full_model_seed42_metrics["iou"]],
                "accuracy": [full_model_seed42_metrics["accuracy"]]}

for seed in EXTRA_SEEDS:
    print(f"--- Training full model with seed {seed} ---")
    m = train_full_run(BUSIMultiTaskNet, f"fullmodel_seed{seed}", seed=seed)
    metrics = evaluate_multitask_model_on_test(m, test_loader)
    print(f"  seed {seed}: dice={metrics['dice']:.4f} iou={metrics['iou']:.4f} accuracy={metrics['accuracy']:.4f}")
    for k in seed_results:
        seed_results[k].append(metrics[k])

print(f"\n=== Statistical Summary (mean +/- std over {len(seed_results['dice'])} seeds) ===")
significance_summary = {}
for k, values in seed_results.items():
    mean_v, std_v = float(np.mean(values)), float(np.std(values))
    significance_summary[k] = {"mean": mean_v, "std": std_v, "values": values}
    print(f"{k:<10}: {mean_v:.4f} +/- {std_v:.4f}")

with open("outputs/statistical_significance.json", "w") as f:
    json.dump(significance_summary, f, indent=2)

print('\nFor your paper, report as e.g.: "Dice = {:.3f} +/- {:.3f}"'.format(
    significance_summary["dice"]["mean"], significance_summary["dice"]["std"]))


## Done — Notebook 3

Files saved to `outputs/`:
- `ablation_results.json` — full model vs. No-CBAM vs. No-Mask-Guidance
- `statistical_significance.json` — mean ± std across random seeds
- Model checkpoints for each variant (`best_fullmodel_seed*.pt`,
  `best_ablation_*.pt`)

You now have everything from all 3 notebooks — combine the tables and
figures from Notebooks 1, 2, and 3 into your paper's results section.
